# KW — Build Semantic Model

Builds a Direct Lake semantic model on top of the `KW_lh` Lakehouse, covering every fact and dimension table created in `01_create_schema`.

The notebook:
1. Generates a Direct Lake model from the lakehouse tables
2. Renames every table and column for business-friendly display
3. Hides surrogate-key columns and sets data categories (address, city, lat/lon, etc.)
4. Wires up every star-schema relationship (and the two header→detail links)
5. Creates a baseline set of measures across sales, operations, marketing and ESG
6. Defines perspectives so each role sees only what they care about

Run this once after `01_create_schema` and any initial load. Re-running with `OVERWRITE = True` rebuilds from scratch.


## 1. Install dependencies

`semantic-link-labs` ships the Tabular Object Model (TOM) wrapper used to script the model.


In [ ]:
%pip install semantic-link-labs

## 2. Parameters

In [ ]:
%pip install -q semantic-link-labs

In [ ]:
import sempy_labs as labs
from sempy_labs.tom import connect_semantic_model

DATASET   = "KWretail"            # semantic-model name
LAKEHOUSE = "KW_LH"               # source lakehouse
WORKSPACE = None                  # None = current workspace
OVERWRITE = True                  # drop & recreate if it already exists


## 3. Generate the Direct Lake semantic model

Picks up every fact / dimension table from the lakehouse in a single call. Direct Lake keeps the data in OneLake — no import, no scheduled refresh.


In [ ]:
DIMENSIONS = [
    "dim_warehouse", "dim_store", "dim_fridge", "dim_supplier", "dim_product",
    "dim_truck", "dim_driver", "dim_route", "dim_forklift", "dim_employee",
    "dim_customer", "dim_campaign", "dim_promotion", "dim_date",
]
FACTS = [
    "fact_store_stock", "fact_inwards_goods", "fact_shipments", "fact_shipment_lines",
    "fact_pick_events", "fact_trips", "fact_sales", "fact_sale_lines",
    "fact_employee_skills", "fact_performance_kpis", "fact_training_records",
    "fact_campaign_performance", "fact_energy_consumption", "fact_wastage",
]
ALL_TABLES = DIMENSIONS + FACTS

labs.directlake.generate_direct_lake_semantic_model(
    dataset=DATASET,
    tables=ALL_TABLES,
    source=LAKEHOUSE,
    source_type      = "Lakehouse",
    source_workspace = WORKSPACE,
    workspace=WORKSPACE,
    refresh=True,
)
print(f"Created semantic model '{DATASET}' with {len(ALL_TABLES)} tables.")


## 4. Friendly table & column names

We rename tables explicitly (drop the `dim_`/`fact_` prefixes, title-case), then auto-prettify every column with a small helper that handles the common abbreviations (ID, KG, AUD, kWh, KPI, SKU, CTR, %, …).


In [ ]:
TABLE_RENAMES = {
    "dim_warehouse": "Warehouse",
    "dim_store": "Store",
    "dim_fridge": "Fridge",
    "dim_supplier": "Supplier",
    "dim_product": "Products",
    "dim_truck": "Truck",
    "dim_driver": "Driver",
    "dim_route": "Route",
    "dim_forklift": "Forklift",
    "dim_employee": "Employee",
    "dim_customer": "Customer",
    "dim_campaign": "Campaign",
    "dim_promotion": "Promotion",
    "dim_date": "Date",
    "fact_store_stock": "Store Stock",
    "fact_inwards_goods": "Inwards Goods",
    "fact_shipments": "Shipments",
    "fact_shipment_lines": "Shipment Lines",
    "fact_pick_events": "Pick Events",
    "fact_trips": "Trips",
    "fact_sales": "Sales",
    "fact_sale_lines": "Sale Lines",
    "fact_employee_skills": "Employee Skills",
    "fact_performance_kpis": "Performance KPIs",
    "fact_training_records": "Training Records",
    "fact_campaign_performance": "Campaign Performance",
    "fact_energy_consumption": "Energy Consumption",
    "fact_wastage": "Wastage",
}

REPLACEMENTS = [
    (" Id", " ID"), ("Id ", "ID "), ("Yyyymm", "YYYYMM"),
    (" Kg", " kg"), ("Kmh", "km/h"), (" Aud", " AUD"),
    ("Kwh", "kWh"), ("Sqm", "sqm"), ("Sku", "SKU"),
    ("Kpi", "KPI"), ("Pct", "%"), ("Ctr", "CTR"),
    ("Rego", "Rego"), ("Dob", "DOB"),
]

def prettify(name: str) -> str:
    s = name.replace("_", " ").title()
    for old, new in REPLACEMENTS:
        s = s.replace(old, new)
    if s.endswith(" Id"): s = s[:-3] + " ID"
    if s == "Id": s = "ID"
    return s

with connect_semantic_model(dataset=DATASET, readonly=False, workspace=WORKSPACE) as tom:
    # Rename columns first (under original table names)
    for tbl_old in list(TABLE_RENAMES.keys()):
        for col in tom.model.Tables[tbl_old].Columns:
            if col.Type.ToString() == "RowNumber":
                continue
            col.Name = prettify(col.Name)
    # Then rename tables
    for tbl_old, tbl_new in TABLE_RENAMES.items():
        tom.model.Tables[tbl_old].Name = tbl_new

print("Tables and columns renamed.")


## 5. Hide surrogate keys and set data categories

Surrogate keys stay in the model (needed for relationships) but are hidden from report builders. Geographic and address columns get the right data categories so Power BI's map visuals work out of the box.


In [ ]:
HIDE_KEY_COLUMNS = {
    "Warehouse":             ["Manager Employee ID"],
    "Store":                 ["Manager Employee ID"],
    "Fridge":                ["Store ID"],
    "Products":              ["Supplier ID"],
    "Truck":                 ["Home Warehouse ID"],
    "Driver":                ["Home Warehouse ID", "Employee ID"],
    "Route":                 ["Origin Warehouse ID", "Destination Store ID"],
    "Forklift":              ["Warehouse ID"],
    "Employee":              ["Location ID", "Manager Employee ID"],
    "Promotion":             ["Product ID"],
    "Store Stock":           ["Store ID", "Product ID"],
    "Inwards Goods":         ["Warehouse ID", "Supplier ID", "Product ID"],
    "Shipments":             ["Warehouse ID", "Store ID"],
    "Shipment Lines":        ["Shipment ID", "Product ID"],
    "Pick Events":           ["Warehouse ID", "Worker Employee ID", "Destination Store ID"],
    "Trips":                 ["Truck ID", "Driver ID", "Route ID"],
    "Sales":                 ["Store ID", "Customer ID", "Salesperson Employee ID"],
    "Sale Lines":            ["Sale ID", "Product ID"],
    "Employee Skills":       ["Employee ID"],
    "Performance KPIs":      ["Employee ID"],
    "Training Records":      ["Employee ID"],
    "Campaign Performance":  ["Campaign ID"],
    "Energy Consumption":    ["Store ID"],
    "Wastage":               ["Store ID", "Product ID"],
}

In [ ]:
DATA_CATEGORIES = {
    "Warehouse": {
        "Address": "Address", "City": "City", "State": "StateOrProvince",
        "Latitude": "Latitude", "Longitude": "Longitude",
    },
    "Store": {
        "Address": "Address", "City": "City", "State": "StateOrProvince",
        "Postcode": "PostalCode",
        "Latitude": "Latitude", "Longitude": "Longitude",
    },
    "Customer": {
        "Postcode": "PostalCode", "State": "StateOrProvince",
    },
    "Supplier": {
        "Country": "Country",
    },
}

with connect_semantic_model(dataset=DATASET, readonly=False, workspace=WORKSPACE) as tom:
    for tbl, cols in HIDE_KEY_COLUMNS.items():
        for c in cols:
            try:
                tom.model.Tables[tbl].Columns[c].IsHidden = True
            except Exception as ex:
                print(f"  skip hide {tbl}[{c}]: {ex}")
    for tbl, cats in DATA_CATEGORIES.items():
        for c, cat in cats.items():
            try:
                tom.model.Tables[tbl].Columns[c].DataCategory = cat
            except Exception as ex:
                print(f"  skip cat  {tbl}[{c}]: {ex}")

print("Keys hidden and data categories applied.")


## 6. Relationships

Every fact-to-dimension and dimension-to-dimension link in the star schema, plus the two header→detail relationships for shipments and sales.

`Receipt Date`, `Ship Date`, `Sale Date`, `Pick Date`, and the campaign / energy / wastage `Date` columns all join to **Date[Date]**.


In [ ]:
RELATIONSHIPS = [
    # Dimension-to-dimension
    ("Fridge",    "Store ID",            "Store",     "Store ID"),
    ("Products",  "Supplier ID",         "Supplier",  "Supplier ID"),
    ("Truck",     "Home Warehouse ID",   "Warehouse", "Warehouse ID"),
    ("Driver",    "Home Warehouse ID",   "Warehouse", "Warehouse ID"),
    ("Route",     "Origin Warehouse ID", "Warehouse", "Warehouse ID"),
    ("Route",     "Destination Store ID","Store",     "Store ID"),
    ("Forklift",  "Warehouse ID",        "Warehouse", "Warehouse ID"),
    ("Promotion", "Product ID",          "Products",   "Product ID"),

    # Store Stock
    ("Store Stock", "Store ID",   "Store",   "Store ID"),
    ("Store Stock", "Product ID", "Products", "Product ID"),

    # Inwards Goods
    ("Inwards Goods", "Warehouse ID",  "Warehouse", "Warehouse ID"),
    ("Inwards Goods", "Supplier ID",   "Supplier",  "Supplier ID"),
    #("Inwards Goods", "Product ID",    "Products",   "Product ID"),
    ("Inwards Goods", "Receipt Date",  "Date",      "Date"),

    # Shipments + lines
    ("Shipments",      "Warehouse ID", "Warehouse", "Warehouse ID"),
    ("Shipments",      "Store ID",     "Store",     "Store ID"),
    ("Shipments",      "Ship Date",    "Date",      "Date"),
    ("Shipment Lines", "Shipment ID",  "Shipments", "Shipment ID"),
    ("Shipment Lines", "Product ID",   "Products",   "Product ID"),

    # Pick Events
    ("Pick Events", "Warehouse ID",          "Warehouse", "Warehouse ID"),
    ("Pick Events", "Worker Employee ID",    "Employee",  "Employee ID"),
    ("Pick Events", "Destination Store ID",  "Store",     "Store ID"),
    ("Pick Events", "Pick Date",             "Date",      "Date"),

    # Trips
    ("Trips", "Truck ID",  "Truck",  "Truck ID"),
    #("Trips", "Driver ID", "Driver", "Driver ID"),
    #("Trips", "Route ID",  "Route",  "Route ID"),

    # Sales + lines
    ("Sales",      "Store ID",                  "Store",    "Store ID"),
    ("Sales",      "Customer ID",               "Customer", "Customer ID"),
    ("Sales",      "Salesperson Employee ID",   "Employee", "Employee ID"),
    ("Sales",      "Sale Date",                 "Date",     "Date"),
    ("Sale Lines", "Sale ID",                   "Sales",    "Sale ID"),
    ("Sale Lines", "Product ID",                "Products",  "Product ID"),

    # HR
    ("Employee Skills",  "Employee ID", "Employee", "Employee ID"),
    ("Performance KPIs", "Employee ID", "Employee", "Employee ID"),
    ("Training Records", "Employee ID", "Employee", "Employee ID"),

    # Marketing
    ("Campaign Performance", "Campaign ID", "Campaign", "Campaign ID"),
    ("Campaign Performance", "Date",        "Date",     "Date"),

    # ESG
    ("Energy Consumption", "Store ID", "Store",   "Store ID"),
    ("Energy Consumption", "Date",     "Date",    "Date"),
    ("Wastage",            "Store ID", "Store",   "Store ID"),
    ("Wastage",            "Product ID","Products","Product ID"),
    ("Wastage",            "Date",     "Date",    "Date"),
]

with connect_semantic_model(dataset=DATASET, readonly=False, workspace=WORKSPACE) as tom:
    for from_t, from_c, to_t, to_c in RELATIONSHIPS:
        try:
            tom.add_relationship(
                from_table=from_t, from_column=from_c,
                to_table=to_t,     to_column=to_c,
                from_cardinality="Many", to_cardinality="One",
                cross_filtering_behavior="OneDirection",
                is_active=True,
            )
        except Exception as ex:
            print(f"  skip rel {from_t}[{from_c}] -> {to_t}[{to_c}]: {ex}")

print(f"Created {len(RELATIONSHIPS)} relationships.")


## 7. Measures

A baseline measure set across sales, lines, wastage, energy, campaigns, trips, picks, inwards, shipments and HR. Display folders keep the field list tidy.


In [ ]:
MEASURES = [
    # Sales
    ("Sales", "Total Sales",    "SUM ( Sales[Total] )",                                "\"$\"#,0",     "Sales"),
    ("Sales", "Total Subtotal", "SUM ( Sales[Subtotal] )",                             "\"$\"#,0",     "Sales"),
    ("Sales", "Total Tax",      "SUM ( Sales[Tax Total] )",                            "\"$\"#,0",     "Sales"),
    ("Sales", "Sale Count",     "DISTINCTCOUNT ( Sales[Sale ID] )",                    "#,0",          "Sales"),
    ("Sales", "Customer Count", "DISTINCTCOUNT ( Sales[Customer ID] )",                "#,0",          "Sales"),
    ("Sales", "Avg Sale Value", "DIVIDE ( [Total Sales], [Sale Count] )",              "\"$\"#,0.00", "Sales"),

    # Sale Lines
    ("Sale Lines", "Total Profit",  "SUM ( \u0027Sale Lines\u0027[Profit] )",        "\"$\"#,0",    "Profitability"),
    ("Sale Lines", "Units Sold",    "SUM ( \u0027Sale Lines\u0027[Qty] )",           "#,0",          "Profitability"),
    ("Sale Lines", "Profit Margin %","DIVIDE ( [Total Profit], [Total Sales] )",        "0.0%",        "Profitability"),

    # Wastage
    ("Wastage", "Wastage Quantity", "SUM ( Wastage[Qty] )",            "#,0",          "Wastage"),
    ("Wastage", "Wastage Cost",     "SUM ( Wastage[Cost AUD] )",       "\"$\"#,0",   "Wastage"),

    # Energy
    ("Energy Consumption", "Total kWh",        "SUM ( \u0027Energy Consumption\u0027[kWh] )",       "#,0",         "Energy"),
    ("Energy Consumption", "Energy Cost AUD",  "SUM ( \u0027Energy Consumption\u0027[Cost AUD] )",  "\"$\"#,0",  "Energy"),

    # Campaign Performance
    ("Campaign Performance", "Impression Count",  "SUM ( \u0027Campaign Performance\u0027[Impressions] )",   "#,0",           "Marketing"),
    ("Campaign Performance", "Click Count",       "SUM ( \u0027Campaign Performance\u0027[Clicks] )",        "#,0",           "Marketing"),
    ("Campaign Performance", "Num Conversions",  "SUM ( \u0027Campaign Performance\u0027[Conversions] )",   "#,0",           "Marketing"),
    ("Campaign Performance", "Revenue",  "SUM ( \u0027Campaign Performance\u0027[Revenue AUD] )",   "\"$\"#,0",    "Marketing"),
    ("Campaign Performance", "CTR %",        "DIVIDE ( [Clicks], [Impressions] )",                        "0.00%",         "Marketing"),
    ("Campaign Performance", "Conversion Rate %","DIVIDE ( [Conversions], [Clicks] )",                    "0.00%",         "Marketing"),

    # Trips
    ("Trips", "Trip Count",          "DISTINCTCOUNT ( Trips[Trip ID] )",                  "#,0",     "Logistics"),
    ("Trips", "Avg Trip Hours",      "AVERAGE ( Trips[Actual Duration Hours] )",          "0.0",     "Logistics"),
    ("Trips", "Avg Fuel L per 100km","AVERAGE ( Trips[Fuel L Per 100Km] )",               "0.0",     "Logistics"),

    # Pick Events
    ("Pick Events", "Num Items Picked",   "SUM ( \u0027Pick Events\u0027[Items Picked] )",                          "#,0", "Warehouse Ops"),
    ("Pick Events", "Total Errors",   "SUM ( \u0027Pick Events\u0027[Errors] )",                                "#,0", "Warehouse Ops"),
    ("Pick Events", "Avg Pick Rate",  "AVERAGE ( \u0027Pick Events\u0027[Pick Rate Items Per Hour] )",          "#,0", "Warehouse Ops"),
    ("Pick Events", "Error Rate %",   "DIVIDE ( [Total Errors], [Items Picked] )",                                "0.00%","Warehouse Ops"),

    # Inwards Goods
    ("Inwards Goods", "Qty Received",  "SUM ( \u0027Inwards Goods\u0027[Qty In] )",                                                              "#,0",         "Warehouse Ops"),
    ("Inwards Goods", "Inwards Cost",  "SUMX ( \u0027Inwards Goods\u0027, \u0027Inwards Goods\u0027[Qty In] * \u0027Inwards Goods\u0027[Unit Cost] )", "\"$\"#,0", "Warehouse Ops"),

    # Shipments
    ("Shipments", "Shipment Count",  "DISTINCTCOUNT ( Shipments[Shipment ID] )",     "#,0",  "Logistics"),
    ("Shipments", "Shipment Weight kg","SUM ( Shipments[Total Weight kg] )",        "#,0",  "Logistics"),

    # HR
    ("Performance KPIs", "Avg KPI Score",   "AVERAGE ( \u0027Performance KPIs\u0027[KPI Score] )",              "0.00",   "Workforce"),
    ("Performance KPIs", "Days of Absence","SUM ( \u0027Performance KPIs\u0027[Absenteeism Days] )",           "#,0",    "Workforce"),
    ("Training Records","Trainings Completed","CALCULATE ( COUNTROWS ( \u0027Training Records\u0027 ), \u0027Training Records\u0027[Status] = \"Completed\" )", "#,0", "Workforce"),
]

with connect_semantic_model(dataset=DATASET, readonly=False, workspace=WORKSPACE) as tom:
    for tbl, name, expr, fmt, folder in MEASURES:
        try:
            tom.add_measure(
                table_name=tbl, measure_name=name, expression=expr,
                format_string=fmt, display_folder=folder,
            )
        except Exception as ex:
            print(f"  skip measure {name}: {ex}")

print(f"Created {len(MEASURES)} measures.")


## 8. Perspectives

Five role-oriented perspectives. A user opening a report against the model can switch perspectives to filter the field list down to just the tables relevant to their job.


In [ ]:
PERSPECTIVES = {
    "Retail": [
        "Store", "Products", "Customer", "Promotion", "Sales", "Sale Lines",
        "Store Stock", "Date",
    ],
    "Operations": [
        "Warehouse", "Truck", "Driver", "Route", "Forklift", "Trips",
        "Pick Events", "Inwards Goods", "Shipments", "Shipment Lines",
        "Supplier", "Products", "Date",
    ],
    "Marketing": [
        "Campaign", "Campaign Performance", "Customer", "Promotion",
        "Products", "Date",
    ],
    "Workforce": [
        "Employee", "Employee Skills", "Performance KPIs", "Training Records",
        "Warehouse", "Store", "Date",
    ],
    "Sustainability": [
        "Energy Consumption", "Wastage", "Store", "Fridge", "Products", "Date",
    ],
}

with connect_semantic_model(dataset=DATASET, readonly=False, workspace=WORKSPACE) as tom:
    for p, tables in PERSPECTIVES.items():
        try:
            tom.add_perspective(perspective_name=p)
        except Exception as ex:
            print(f"  perspective {p} already exists or error: {ex}")
        for t in tables:
            try:
                tbl_obj = tom.model.Tables[t]
                tom.add_to_perspective(object=tbl_obj, perspective_name=p)
            except Exception as ex:
                print(f"  skip add {t} -> {p}: {ex}")

print(f"Created {len(PERSPECTIVES)} perspectives.")


## 9. Refresh & summary

A final Direct Lake refresh picks up column statistics so the model is query-ready. Then list what we built.


In [ ]:
labs.refresh_semantic_model(dataset=DATASET, workspace=WORKSPACE, refresh_type="full")

with connect_semantic_model(dataset=DATASET, readonly=True, workspace=WORKSPACE) as tom:
    n_tables   = sum(1 for _ in tom.model.Tables)
    n_columns  = sum(1 for t in tom.model.Tables for _ in t.Columns)
    n_measures = sum(1 for t in tom.model.Tables for _ in t.Measures)
    n_rels     = sum(1 for _ in tom.model.Relationships)
    n_persp    = sum(1 for _ in tom.model.Perspectives)
    print(f"Tables:        {n_tables}")
    print(f"Columns:       {n_columns}")
    print(f"Measures:      {n_measures}")
    print(f"Relationships: {n_rels}")
    print(f"Perspectives:  {n_persp}")
